# Simulation Data Preprocessing

The simulations record the trajectories (a series of x,y,z coordinates, with associated amplitude of each point) of products Fission experiments in ATTPC.

This notebook preprocesses Adam's simulation data as following:

1. Imports the events from the H5 file.
2. Samples events upto 512 points.
3. Filters events by angle/radial seperation.
4. Scales/normalizes the events.

Existing file naming convention:
- Data with space change: yesSC
- Data without space change: noSC

### A. User-Desired Settings

The isotope used in this experiment is Fission.

In [ ]:
ISOTOPE = 'Fission'

The neural network model requires a fixed number of inputs. Whereas the actual events comprise different number of points, we will select exactly 512 points (may be redundant) as final inputs of each event.

In [ ]:
sample_size = 512

We create a folder named "test" to store the outputs.

In [ ]:
dir_name = 'fission_data/'

### B. Import Libraries

In [ ]:
import h5py
import numpy as np
import tqdm
import math
import random
import copy
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits import mplot3d

### C. Misc. Functions

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def visualize(arr, r, c):
    """
    Visualize 4D fission point clouds: (x, y, z, log10(charge))
    Each subplot shows one event with its own colorbar.
    """
    fig = plt.figure(figsize=(16, r * 4))

    for i in range(min(r * c, len(arr))):
        ax = fig.add_subplot(r, c, i + 1, projection='3d')
        cloud = arr[i]

        # Split coordinates and color dimension
        xs, ys, zs, colors = cloud[:, 0], cloud[:, 1], cloud[:, 2], cloud[:, 3]

        # Scatter plot (cool colormap)
        sc = ax.scatter(xs, zs, ys, c=colors, cmap='cool', s=1, vmin=0, vmax=4)
        ax.set_title(f'Fission Event {i + 1}', fontsize=10)

        # Axis labels
        ax.set_xlabel('x', fontsize=9)
        ax.set_ylabel('z', fontsize=9)
        ax.set_zlabel('y', fontsize=9)

        # Fixed axis limits (based on normalized data)
        ax.set_xlim(-1, 1)
        ax.set_ylim(1, 3)
        ax.set_zlim(-1, 1)

        # Aesthetic tweaks
        ax.grid(True)
        ax.set_facecolor((0.97, 0.97, 0.97))

        # Add individual colorbar for each subplot
        cbar = plt.colorbar(sc, ax=ax, shrink=0.7, pad=0.05)
        cbar.set_label('log₁₀(charge)', rotation=270, labelpad=10)

    plt.tight_layout()
    plt.show()


In [ ]:
def visualize_unscaled(arr, r, c):
    """
    Visualize 4D fission point clouds: (x, y, z, log10(charge))
    Each subplot shows one event with its own colorbar.
    """
    fig = plt.figure(figsize=(16, r * 4))

    for i in range(min(r * c, len(arr))):
        ax = fig.add_subplot(r, c, i + 1, projection='3d')
        cloud = arr[i]

        # Split coordinates and color dimension
        xs, ys, zs, colors = cloud[:, 0], cloud[:, 1], cloud[:, 2], cloud[:, 3]

        # Scatter plot (cool colormap)
        sc = ax.scatter(xs, zs, ys, c=colors, cmap='cool', s=1, vmin=0, vmax=4)
        ax.set_title(f'Fission Event {i + 1}', fontsize=10)

        # Axis labels
        ax.set_xlabel('x', fontsize=9)
        ax.set_ylabel('z', fontsize=9)
        ax.set_zlabel('y', fontsize=9)

        # Fixed axis limits (based on normalized data)
        ax.set_xlim(-250, 250)
        ax.set_ylim(0, 1000)
        ax.set_zlim(-250, 250)

        # Aesthetic tweaks
        ax.grid(True)
        ax.set_facecolor((0.97, 0.97, 0.97))

        # Add individual colorbar for each subplot
        cbar = plt.colorbar(sc, ax=ax, shrink=0.7, pad=0.05)
        cbar.set_label('log₁₀(charge)', rotation=270, labelpad=10)

    plt.tight_layout()
    plt.show()

## -- Data Preprocessing Steps --

## 1. Import the events from the H5 file

In [ ]:
def import_data(file):
    event_ids = list(file.keys())
    
    event_ids = sorted(
    file.keys(),
    key=lambda x: int(x.split('[')[1].split(']')[0]))
    num_of_event = len(event_ids)
    ev_lens = np.zeros(num_of_event, int)
    
    for i in range(num_of_event):
        event_id = event_ids[i]
        event = file[event_id]['HitArray']
        ev_lens[i] = len(file[event_id]['HitArray'])
    evlen_path = dir_name + ISOTOPE + '_sim' + '_XYZAPPE_ev_lens'
    np.save(evlen_path, ev_lens)

    data = np.zeros((num_of_event, np.max(ev_lens), 7), float) # XYZAPPE
    for n in tqdm.tqdm(range(num_of_event)):
        event_id = event_ids[n]
        event = file[event_id]['HitArray']
        #converting event into an array
        for i,e in enumerate(event):
            instant = np.array(list(e))
            data[n][i][0:3] = np.array(instant[0:3]) # x,y,z
            data[n][i][3] = np.array(instant[4]) # amplitude
            data[n][i][4] = np.array(instant[5])-1 # particleID--lower index to start at 0
            data[n][i][5] = np.arange(1,np.max(ev_lens)+1)[i] # pointID
            data[n][i][-1] = float(n) # eventID
    data_path = dir_name  + ISOTOPE + '_sim' + '_XYZAPPE'
    np.save(data_path, data)
    
    return ev_lens, data

In [ ]:
file = h5py.File(dir_name + 'raw/' + 'output_digi02.h5', 'r')

ev_lens, data = import_data(file)

When running this notebook the second time, simply reload the data (instead of spending 10 min to repeat the step above).

In [ ]:
ev_lens = np.load(dir_name + ISOTOPE + '_sim_yesSC_XYZAPPE_ev_lens.npy')
data = np.load(dir_name + ISOTOPE + '_sim_yesSC_XYZAPPE.npy')

In [ ]:
print("Initial data shape: ", data.shape)

## 2. Samples events upto 512 points

In [ ]:
evlen_path = dir_name + ISOTOPE + '_sim_yesSC_XYZAPPE_ev_lens.npy'
data_path = dir_name + ISOTOPE + '_sim_yesSC_XYZAPPE.npy'
data_noNull = np.load(data_path)
num_of_event = len(data_noNull)
max_ev_len = len(data_noNull[0])
ev_lens = np.load(evlen_path)
data_sampled = np.zeros((num_of_event, sample_size, 7), float) #XYZAPPE

In [ ]:
for n in tqdm.tqdm(range(num_of_event)):
    ev_len = ev_lens[n]
    if ev_len >= sample_size:
        data_sampled[n,:sample_size,:] = data_noNull[n,:sample_size,:]
    else:
        data_sampled[n,:ev_len,:] = data_noNull[n,:ev_len,:]
        need = sample_size - ev_len
        random_points = np.random.choice(range(ev_len), need, replace=True if need > ev_len else False) 
        instant = ev_len
        for r in random_points:
            data_sampled[n,instant,:] = data_noNull[n,r,:] 
            instant += 1

data_path = dir_name + ISOTOPE + '_sim_yesSC_sampled_size' + str(sample_size)
np.save(data_path, data_sampled)
print("Sampled data shape: ", data_sampled.shape)

### 2.1 Get XYZC

In [ ]:
data = np.load(dir_name + ISOTOPE + '_sim_yesSC_sampled_size' + str(sample_size) + '.npy')
new_data = data[:,:, [0,1,2,3]]
data_path = dir_name + ISOTOPE + '_sim_yesSC_sampled_XYZC'
np.save(data_path, new_data)

print("Sampled XYZC data shape: ", new_data.shape)

## 3. Filters events by angle/radial seperation

In [ ]:
unfiltered_data = new_data

# ── Class definitions ────────────────────────────────────────────────────────
CLASSES = {
    'Small':  (0,   50),
    'Medium': (50,  80),
    'Large':  (80, 999),
}
CLASS_COLORS = {'Small': 'red', 'Medium': 'orange', 'Large': 'green'}


def max_radius_fixed_z(event, z_min=0, z_max=100):
    x, y, z = event[:, 0], event[:, 1], event[:, 2]
    
    # remove zero-padded rows
    real = (z > 1e-3) | (np.abs(x) > 1e-3) | (np.abs(y) > 1e-3)
    x, y, z = x[real], y[real], z[real]
    
    if len(z) == 0:
        return 0.0
    
    # fixed z window
    in_region = (z >= z_min) & (z <= z_max)
    
    if in_region.sum() == 0:
        return 0.0
     
    return float(np.sqrt(x[in_region]**2 + y[in_region]**2).max())


def classify(r):
    for name, (lo, hi) in CLASSES.items():
        if lo <= r < hi:
            return name
    return 'Large'


# ── Compute and assign classes ───────────────────────────────────────────────
r_max  = np.array([max_radius_fixed_z(unfiltered_data[i]) for i in range(len(unfiltered_data))])
labels = np.array([classify(r) for r in r_max])
 
for name in CLASSES:
    print(f"{name:8s}: {np.sum(labels == name):4d} events ({100*np.mean(labels == name):.1f}%)")

In [ ]:
N_ROWS = 3
N_COLS = 3
N_SHOW = N_ROWS * N_COLS
rng = np.random.default_rng(10)

for class_name in CLASSES:
    idx_pool = np.where(labels == class_name)[0]
    if len(idx_pool) == 0:
        continue

    chosen = rng.choice(idx_pool, size=min(N_SHOW, len(idx_pool)), replace=False)
    color  = CLASS_COLORS[class_name]

    # DOUBLE columns for XZ + YZ
    fig, axes = plt.subplots(N_ROWS, N_COLS * 2, figsize=(5 * N_COLS * 2, 4 * N_ROWS))

    fig.suptitle(f'Class: {class_name}  ({len(idx_pool)} events)',
                 fontsize=13, fontweight='bold', color=color)

    for i, idx in enumerate(chosen):
        row = i // N_COLS
        col = i % N_COLS

        evt = unfiltered_data[idx]

        # filter real points
        real = (evt[:, 2] > 1e-3) | (np.abs(evt[:, 0]) > 1e-3) | (np.abs(evt[:, 1]) > 1e-3)
        evt = evt[real]

        # ---- XZ projection ----
        ax_xz = axes[row, 2*col]
        ax_xz.scatter(evt[:, 2], evt[:, 0], s=3, alpha=0.4, color=color)
        ax_xz.set_xlim([0, 1000])
        ax_xz.set_ylim([-250, 250])
        ax_xz.set_xlabel('Z (mm)')
        ax_xz.set_ylabel('X (mm)')
        ax_xz.set_title(f'Event {idx} (XZ)', fontsize=8)

        # ---- YZ projection ----
        ax_yz = axes[row, 2*col + 1]
        ax_yz.scatter(evt[:, 2], evt[:, 1], s=3, alpha=0.4, color=color)
        ax_yz.set_xlim([0, 1000])
        ax_yz.set_ylim([-250, 250])
        ax_yz.set_xlabel('Z (mm)')
        ax_yz.set_ylabel('Y (mm)')
        ax_yz.set_title(f'Event {idx} (YZ)', fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
good_mask = labels == 'Large'
good_events = unfiltered_data[good_mask]
good_indices = np.where(good_mask)[0]

data_path = dir_name + ISOTOPE + '_sim_yesSC_sampled_XYZC_filtered'
indices_path = dir_name + ISOTOPE + '_sim_yesSC_sampled_XYZC_filtered_indices'

np.save(data_path, good_events)
np.save(indices_path, good_indices)

print("Filtered data shape: ", good_events.shape)
print("Filtered indices shape: ", good_indices.shape)

## 4. Normalize

In [ ]:
data_path = dir_name + ISOTOPE + '_sim_yesSC_sampled_XYZC_filtered.npy'
data_sampled = np.load(data_path)
data_scaled = data_sampled

data_sampled[:,:,3] = np.where(data_sampled[:,:,3] > 0, data_sampled[:,:,3], 1)
data_sampled[:,:,3] = np.where(data_sampled[:,:,3] < 10000, data_sampled[:,:,3], 10000)   
data_scaled[:,:,3] = np.log10(data_sampled[:,:,3])

for n in range(3):
    if n == 0 or n == 1:
        data_scaled[:,:,n] /= 250
    else:
        data_scaled[:,:,n] = data_scaled[:,:,n]/500 + 1

data_path = dir_name + ISOTOPE + '_sim_yesSC_sampled_XYZC_filtered_scaled.npy'
np.save(data_path, data_scaled)

In [ ]:
print("Final data shape: ", data_scaled.shape)
print("Final indices shape: ", good_indices.shape)

In [ ]:
print("X-range: [", data_scaled[:,:,0].min(), ",",data_scaled[:,:,0].max(), "]")
print("Y-range: [", data_scaled[:,:,1].min(), ",",data_scaled[:,:,1].max(), "]")
print("Z-range: [", data_scaled[:,:,2].min(), ",",data_scaled[:,:,2].max(), "]")
print("C-range: [", data_scaled[:,:,3].min(), ",",data_scaled[:,:,3].max(), "]")

## 5. Visualize and Inspect

In [ ]:
data_path = dir_name + ISOTOPE + '_sim_yesSC_sampled_XYZC_filtered_scaled.npy'
final_data = np.load(data_path)
visualize(data_sampled, 3, 3)

In [ ]:
print("Shape:", data_sampled.shape)

# 1. NaNs
print("Contains NaN:", np.isnan(data_sampled).any())

# 2. Infs
print("Contains Inf:", np.isinf(data_sampled).any())

# 3. Min / Max values
print("Min value:", np.min(data_sampled))
print("Max value:", np.max(data_sampled))

# 4. Check extreme magnitudes
print("Max absolute value:", np.max(np.abs(data_sampled)))

# 5. Per-dimension stats
print("\nPer-dimension stats:")
for i in range(data_sampled.shape[-1]):
    col = data_sampled[..., i]
    print(f"Dim {i}: min={col.min():.4f}, max={col.max():.4f}, mean={col.mean():.4f}, std={col.std():.4f}")
